# 🔩 Steel Defect Detection — Live Demo
**3-model ensemble: U-Net + FPN + DeepLabV3+**

Run all cells to start the public web demo. The URL at the bottom works for anyone in the world.


In [1]:
# Install all dependencies
!pip install -q segmentation-models-pytorch albumentations fastapi uvicorn python-multipart huggingface_hub nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 2.9 MB/s eta 0:00:00


In [2]:
import os
from huggingface_hub import hf_hub_download

HF_REPO = "NarekGabrielyan/steel-defect-detection"
os.makedirs("checkpoints", exist_ok=True)

files = [
    "checkpoints/resnet34_unet_last.pth",
    "checkpoints/fpn_se_resnext50_last.pth",
    "checkpoints/deeplabv3p_efficientnetb3_last.pth",
]

for f in files:
    local = hf_hub_download(repo_id=HF_REPO, filename=f, local_dir=".")
    print(f"Downloaded: {local}")


checkpoints/resnet34_unet_last.pth: reconstructing file:   0%|          |  0.00B /  294MB            

checkpoints/resnet34_unet_last.pth: downloading bytes:           |  0.00B            

Downloaded: /content/checkpoints/resnet34_unet_last.pth


checkpoints/fpn_se_resnext50_last.pth: reconstructing file:   0%|          |  0.00B /  338MB            

checkpoints/fpn_se_resnext50_last.pth: downloading bytes:           |  0.00B            

Downloaded: /content/checkpoints/fpn_se_resnext50_last.pth


checkpoints/deeplabv3p_efficientnetb3_la(…): reconstructing file:   0%|          |  0.00B /  136MB            

checkpoints/deeplabv3p_efficientnetb3_la(…): downloading bytes:           |  0.00B            

Downloaded: /content/checkpoints/deeplabv3p_efficientnetb3_last.pth


In [3]:
# Get the src/ package from GitHub
!git clone --depth 1 https://github.com/gabrielyannarek04-max/steel-defect-detection.git _repo 2>/dev/null || true
import shutil, os
if os.path.exists("_repo/src"):
    shutil.copytree("_repo/src", "src", dirs_exist_ok=True)
    print("src/ package copied.")


src/ package copied.


In [4]:
%%writefile server.py
import os, sys, base64, cv2, numpy as np, torch
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from fastapi.staticfiles import StaticFiles
from fastapi.middleware.cors import CORSMiddleware
import albumentations as A
from albumentations.pytorch import ToTensorV2
sys.path.insert(0, ".")
from src.models import build_model

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MIN_AREAS = [300, 300, 1000, 2000]
CONFIGS = [
    ("unet",          "resnet34",           "checkpoints/resnet34_unet_last.pth"),
    ("fpn",           "se_resnext50_32x4d", "checkpoints/fpn_se_resnext50_last.pth"),
    ("deeplabv3plus", "efficientnet-b3",    "checkpoints/deeplabv3p_efficientnetb3_last.pth"),
]

MODELS = []
for arch, enc, path in CONFIGS:
    try:
        m = build_model(arch, enc, encoder_weights=None, classes=4)
        ckpt = torch.load(path, map_location=DEVICE)
        m.load_state_dict(ckpt.get("model_state_dict", ckpt))
        m.to(DEVICE).eval()
        MODELS.append(m)
        print(f"[{arch}] loaded")
    except Exception as e:
        print(f"[{arch}] failed: {e}")

TRANSFORM = A.Compose([A.Resize(256,1600), A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])
COLORS = [(255,50,50),(50,220,50),(50,100,255),(255,210,50)]

@app.post("/api/predict")
async def predict(file: UploadFile = File(...)):
    data = await file.read()
    arr  = np.frombuffer(data, np.uint8)
    img  = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    vis  = cv2.resize(img, (1600, 256))
    t    = TRANSFORM(image=img)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = sum(torch.sigmoid(m(t).squeeze(0)) for m in MODELS) / len(MODELS)
    raw = (probs > 0.5).cpu().numpy().astype(np.uint8)
    found = []
    overlay = vis.astype(np.float32)
    for c in range(4):
        n, lbl, stats, _ = cv2.connectedComponentsWithStats(raw[c], connectivity=8)
        mask = np.zeros_like(raw[c])
        for i in range(1, n):
            if stats[i, cv2.CC_STAT_AREA] >= MIN_AREAS[c]:
                mask[lbl == i] = 1
        if mask.sum():
            found.append(c+1)
            layer = np.zeros_like(overlay)
            layer[mask==1] = COLORS[c]
            overlay = cv2.addWeighted(overlay, 1.0, layer, 0.55, 0)
    _, buf = cv2.imencode(".jpg", cv2.cvtColor(overlay.astype(np.uint8), cv2.COLOR_RGB2BGR))
    return {"detected_classes": found, "image_base64": "data:image/jpeg;base64," + base64.b64encode(buf).decode()}

app.mount("/", StaticFiles(directory="static", html=True), name="static")


Writing server.py


In [7]:
import os, time, re, urllib.request

print("Cleaning up old files...")
!rm -rf _repo src static server.py server.log lt.log

print("Downloading fresh files from GitHub...")
!git clone -q https://github.com/gabrielyannarek04-max/steel-defect-detection.git _repo
!mv _repo/src .
!mv _repo/static .

print("Writing server.py...")
with open("server.py", "w") as f:
    f.write("""
import os, sys, base64, cv2, numpy as np, torch
from fastapi import FastAPI, UploadFile, File
from fastapi.staticfiles import StaticFiles
from fastapi.middleware.cors import CORSMiddleware
import albumentations as A
from albumentations.pytorch import ToTensorV2
sys.path.insert(0, ".")
from src.models import build_model

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODELS = []
try:
    for arch, enc, path in [("unet", "resnet34", "checkpoints/resnet34_unet_last.pth"),
                            ("fpn", "se_resnext50_32x4d", "checkpoints/fpn_se_resnext50_last.pth"),
                            ("deeplabv3plus", "efficientnet-b3", "checkpoints/deeplabv3p_efficientnetb3_last.pth")]:
        m = build_model(arch, enc, encoder_weights=None, classes=4)
        m.load_state_dict(torch.load(path, map_location=DEVICE).get("model_state_dict", torch.load(path, map_location=DEVICE)))
        MODELS.append(m.to(DEVICE).eval())
except Exception as e:
    print(f"Model load error: {e}")

TRANSFORM = A.Compose([A.Resize(256,1600), A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])
COLORS, MIN_AREAS = [(255,50,50),(50,220,50),(50,100,255),(255,210,50)], [300, 300, 1000, 2000]

@app.post("/api/predict")
async def predict(file: UploadFile = File(...)):
    img = cv2.cvtColor(cv2.imdecode(np.frombuffer(await file.read(), np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    vis = cv2.resize(img, (1600, 256))
    t = TRANSFORM(image=img)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = sum(torch.sigmoid(m(t).squeeze(0)) for m in MODELS) / len(MODELS)
    raw, found, overlay = (probs > 0.5).cpu().numpy().astype(np.uint8), [], vis.astype(np.float32)
    for c in range(4):
        n, lbl, stats, _ = cv2.connectedComponentsWithStats(raw[c], connectivity=8)
        mask = np.zeros_like(raw[c])
        for i in range(1, n):
            if stats[i, cv2.CC_STAT_AREA] >= MIN_AREAS[c]: mask[lbl == i] = 1
        if mask.sum():
            found.append(c+1)
            layer = np.zeros_like(overlay)
            layer[mask==1] = COLORS[c]
            overlay = cv2.addWeighted(overlay, 1.0, layer, 0.55, 0)
    _, buf = cv2.imencode(".jpg", cv2.cvtColor(overlay.astype(np.uint8), cv2.COLOR_RGB2BGR))
    return {"detected_classes": found, "image_base64": "data:image/jpeg;base64," + base64.b64encode(buf).decode()}

app.mount("/", StaticFiles(directory="static", html=True), name="static")
""")

print("Starting FastAPI server...")
!nohup uvicorn server:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &
time.sleep(3)

print("Starting LocalTunnel...")
!npm install -g localtunnel > /dev/null 2>&1
!nohup lt --port 8000 > lt.log 2>&1 &
time.sleep(4)

with open("lt.log", "r") as f:
    logs = f.read()

match = re.search(r"https://[-a-zA-Z0-9]+\.loca\.lt", logs)
if match:
    url = match.group(0)
    print("="*60)
    print(f"  🔥 LIVE PUBLIC URL: {url}")
    try:
        req = urllib.request.urlopen("https://loca.lt/mytunnelpassword")
        print(f"  🔑 ENTER THIS IP ON THE WEBPAGE: {req.read().decode('utf-8')}")
    except: pass
    print("="*60)
else:
    print("Failed to get URL. Server logs:")
    !cat server.log
    print("\nTunnel logs:")
    !cat lt.log

Cleaning up old files...
Writing server.py...
Starting FastAPI server...
Starting LocalTunnel...
  🔥 LIVE PUBLIC URL: https://many-trams-kneel.loca.lt
  🔑 ENTER THIS IP ON THE WEBPAGE: 34.11.102.2
